# SEC EDGAR Raw Data Test

**Phase 1 — No cloud, no Snowflake. Pure HTTP → DataFrame.**

Tests all 3 public SEC EDGAR endpoints:
1. `company_tickers_exchange.json` — full ticker/exchange snapshot
2. `submissions/CIK{cik}.json` — company metadata (SIC, entity type, filings list)
3. `api/xbrl/companyfacts/CIK{cik}.json` — XBRL financial facts

**Mandatory SEC compliance:**
- `User-Agent` header on every request (SEC Developer Resources requirement)
- Rate limited to ≤8 req/s (SEC hard limit is 10 req/s per IP)

In [ ]:
# ── 0. Dependencies ───────────────────────────────────────────────────────────
# Run once if needed:
# !pip install requests pyarrow pandas duckdb

In [ ]:
import os, json, time, threading, sys
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

# ── SEC User-Agent (required by SEC Developer Resources) ─────────────────────
os.environ["SEC_USER_AGENT"] = "n/a prototyping paul.ananth@yahoo.com"

USER_AGENT = os.environ["SEC_USER_AGENT"]
print(f"User-Agent : {USER_AGENT}")
print(f"Date       : {date.today()}")

In [ ]:
# ── Rate limiter (token-bucket, 8 req/s) ─────────────────────────────────────
class RateLimiter:
    """Thread-safe token-bucket. Caps at 8 req/s (20% below SEC's 10 req/s limit)."""
    def __init__(self, rps=8.0):
        self._rps, self._tokens, self._last = rps, rps, time.monotonic()
        self._lock = threading.Lock()
    def acquire(self):
        with self._lock:
            now = time.monotonic()
            self._tokens = min(self._rps, self._tokens + (now - self._last) * self._rps)
            self._last = now
            if self._tokens < 1.0:
                time.sleep((1.0 - self._tokens) / self._rps)
                self._tokens = 0.0
            else:
                self._tokens -= 1.0

# ── HTTP session (User-Agent injected on every request) ──────────────────────
_retry = Retry(total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503])
_session = requests.Session()
_session.headers["User-Agent"] = USER_AGENT
_session.mount("https://", HTTPAdapter(max_retries=_retry))

_limiter = RateLimiter(rps=8.0)

def fetch(url):
    """Rate-limited GET. Returns parsed JSON or None on 404."""
    _limiter.acquire()
    r = _session.get(url, timeout=20)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

print("HTTP client ready  (User-Agent set, rate limiter 8 req/s)")

---
## 1. Ticker Exchange Snapshot
Single daily file — all tickers listed on NYSE and Nasdaq (~10,000 rows).

In [ ]:
raw = fetch("https://www.sec.gov/files/company_tickers_exchange.json")

tickers_df = pd.DataFrame(raw["data"], columns=raw["fields"])
tickers_df["cik"] = tickers_df["cik"].astype(str).str.zfill(10)

print(f"Total rows : {len(tickers_df):,}")
print(f"Exchanges  : {tickers_df['exchange'].value_counts().to_dict()}")
tickers_df.head(10)

In [ ]:
# Filter to NYSE + Nasdaq only
listed = tickers_df[tickers_df["exchange"].isin(["NYSE", "Nasdaq"])].copy()
print(f"NYSE + Nasdaq : {len(listed):,} tickers")
listed.sample(10)

---
## 2. Submissions — Company Metadata
One JSON file per CIK. Contains SIC code, entity type, state of incorporation, and a full filings index.

In [ ]:
# Pilot set — 10 well-known companies
PILOT = [
    ("AAPL",  "0000320193"),
    ("MSFT",  "0000789019"),
    ("AMZN",  "0001018724"),
    ("GOOG",  "0001652044"),
    ("META",  "0001326801"),
    ("TSLA",  "0001318605"),
    ("JPM",   "0000019617"),
    ("JNJ",   "0000200406"),
    ("XOM",   "0000034088"),
    ("BRK-B", "0001067983"),
]

def parse_submission(ticker, cik10, data):
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    dates = filings.get("filingDate", [])

    # Active flag — needs recent 10-K or 10-Q within 2 years
    cutoff = (datetime.today() - timedelta(days=730)).date()
    annual = {"10-K", "10-Q", "10-K/A", "10-Q/A"}
    active = any(
        f in annual and date.fromisoformat(d) >= cutoff
        for f, d in zip(forms, dates)
        if d
    )

    exchanges = list(dict.fromkeys(data.get("exchanges", [])))
    recent_forms = list(dict.fromkeys(forms[:20]))  # first 20 unique

    return {
        "ticker":              ticker,
        "cik":                 cik10,
        "name":                data.get("name"),
        "sic":                 data.get("sic"),
        "sic_description":     data.get("sicDescription"),
        "state_of_inc":        data.get("stateOfIncorporation"),
        "ein":                 data.get("ein"),
        "entity_type":         data.get("entityType"),
        "filer_category":      data.get("category"),
        "exchanges":           ", ".join(exchanges),
        "active":              active,
        "total_filings":       len(forms),
        "recent_form_types":   ", ".join(recent_forms[:8]),
    }

submissions = []
for ticker, cik10 in PILOT:
    data = fetch(f"https://data.sec.gov/submissions/CIK{cik10}.json")
    if data:
        row = parse_submission(ticker, cik10, data)
        submissions.append(row)
        print(f"  {ticker:<8} {row['name'][:45]:<45}  SIC={row['sic']}  active={row['active']}")
    else:
        print(f"  {ticker:<8} 404")

submissions_df = pd.DataFrame(submissions)
print(f"\nFetched {len(submissions_df)} companies")

In [ ]:
submissions_df[[
    "ticker", "name", "sic", "sic_description",
    "state_of_inc", "entity_type", "filer_category", "active"
]]

In [ ]:
# Field coverage
print("Field coverage:")
for col in submissions_df.columns:
    filled = submissions_df[col].notna().sum()
    print(f"  {col:<30} {100*filled/len(submissions_df):5.1f}%")

---
## 3. XBRL Company Facts — Financial Data
The largest payload (~5–15 MB per company). Contains every tagged financial value ever filed with the SEC.

In [ ]:
# Which XBRL concepts to extract
CONCEPTS = {
    "Revenues":                                        "revenue",
    "RevenueFromContractWithCustomerExcludingAssessedTax": "revenue_alt",
    "NetIncomeLoss":                                   "net_income",
    "OperatingIncomeLoss":                             "operating_income",
    "Assets":                                          "total_assets",
    "Liabilities":                                     "total_liabilities",
    "StockholdersEquity":                              "stockholders_equity",
    "LongTermDebt":                                    "long_term_debt",
    "CashAndCashEquivalentsAtCarryingValue":            "cash",
    "NetCashProvidedByUsedInOperatingActivities":       "operating_cf",
    "EarningsPerShareBasic":                           "eps_basic",
    "EarningsPerShareDiluted":                         "eps_diluted",
}

ANNUAL_MIN, ANNUAL_MAX = 355, 375

def period_type(entry):
    if "start" not in entry:
        return "instant"
    try:
        days = (date.fromisoformat(entry["end"]) - date.fromisoformat(entry["start"])).days
        if ANNUAL_MIN <= days <= ANNUAL_MAX: return "annual"
        if 85 <= days <= 100: return "quarterly"
        return "other"
    except ValueError:
        return "other"

def parse_facts(ticker, cik10, data):
    rows = []
    facts_root = data.get("facts", {})
    for ns in ("us-gaap", "ifrs-full"):
        ns_data = facts_root.get(ns, {})
        for concept, short_name in CONCEPTS.items():
            cd = ns_data.get(concept)
            if cd is None: continue
            for unit, entries in cd.get("units", {}).items():
                for e in entries:
                    pt = period_type(e)
                    if pt not in ("annual", "instant"): continue
                    rows.append({
                        "ticker": ticker, "cik": cik10,
                        "concept": short_name, "xbrl_concept": concept,
                        "unit": unit,
                        "end_date": e.get("end"),
                        "start_date": e.get("start"),
                        "value": e.get("val"),
                        "form": e.get("form"),
                        "filed": e.get("filed"),
                        "period_type": pt,
                    })
        if rows: break  # found us-gaap data, skip ifrs
    return rows

# Fetch all 10 companies sequentially (large payloads)
all_facts = []
for ticker, cik10 in PILOT:
    data = fetch(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json")
    if data:
        rows = parse_facts(ticker, cik10, data)
        all_facts.extend(rows)
        print(f"  {ticker:<8} {len(rows):>5} rows")
    else:
        print(f"  {ticker:<8} 404")

facts_df = pd.DataFrame(all_facts)
facts_df["value"] = pd.to_numeric(facts_df["value"], errors="coerce")
facts_df["end_date"] = pd.to_datetime(facts_df["end_date"], errors="coerce")
print(f"\nTotal fact rows: {len(facts_df):,}")

In [ ]:
facts_df.head(10)

---
## 4. Spot Checks — Annual Revenue & Net Income

In [ ]:
# Annual revenue for each company (most recent 5 years)
revenue = (
    facts_df[
        (facts_df["concept"].isin(["revenue", "revenue_alt"])) &
        (facts_df["period_type"] == "annual") &
        (facts_df["unit"] == "USD") &
        (facts_df["form"].isin(["10-K", "10-K/A"]))
    ]
    .sort_values("end_date", ascending=False)
    .groupby("ticker")
    .head(5)
    [["ticker", "end_date", "value", "form", "filed"]]
    .assign(revenue_bn=lambda x: (x["value"] / 1e9).round(2))
)
revenue

In [ ]:
# Pivot: revenue by ticker × fiscal year
pivot = (
    revenue
    .assign(year=revenue["end_date"].dt.year)
    .drop_duplicates(["ticker", "year"])
    .pivot_table(index="ticker", columns="year", values="revenue_bn", aggfunc="first")
    .sort_index()
)
print("Annual Revenue ($B)")
pivot

In [ ]:
# Net income — most recent annual value per company
net_income = (
    facts_df[
        (facts_df["concept"] == "net_income") &
        (facts_df["period_type"] == "annual") &
        (facts_df["unit"] == "USD") &
        (facts_df["form"].isin(["10-K", "10-K/A"]))
    ]
    .sort_values("end_date", ascending=False)
    .groupby("ticker")
    .first()
    .reset_index()
    [["ticker", "end_date", "value"]]
    .assign(net_income_bn=lambda x: (x["value"] / 1e9).round(2))
    .sort_values("net_income_bn", ascending=False)
)
print("Most Recent Annual Net Income ($B)")
net_income

In [ ]:
# AAPL — all available annual facts (what concepts are covered?)
aapl = facts_df[
    (facts_df["ticker"] == "AAPL") &
    (facts_df["period_type"] == "annual") &
    (facts_df["unit"] == "USD")
].copy()

print(f"AAPL annual fact rows: {len(aapl)}")
print("\nConcepts available:")
print(aapl["concept"].value_counts().to_string())

---
## 5. Data Quality Summary

In [ ]:
# Coverage: which companies have which concepts?
coverage = (
    facts_df[
        (facts_df["period_type"] == "annual") &
        (facts_df["unit"] == "USD")
    ]
    .groupby(["ticker", "concept"])
    .size()
    .unstack(fill_value=0)
    .map(lambda x: "Y" if x > 0 else "-")
)
print("Concept coverage per company (Y = has data)")
coverage

In [ ]:
# Overall summary
print("=" * 55)
print("PHASE 1 — RAW DATA SUMMARY")
print("=" * 55)
print(f"  Tickers snapshot  : {len(tickers_df):>7,} rows")
print(f"  NYSE + Nasdaq     : {len(listed):>7,} tickers")
print(f"  Submissions       : {len(submissions_df):>7,} companies")
print(f"  Fact rows total   : {len(facts_df):>7,}")
print(f"  Annual USD facts  : {len(facts_df[(facts_df.period_type=='annual')&(facts_df.unit=='USD')]):>7,}")
print()
print("All SEC compliance checks:")
print(f"  User-Agent header : {USER_AGENT}")
print(f"  Rate limit        : 8 req/s (SEC limit = 10 req/s)")
print("  Sequential ingest : ingest scripts run one at a time")